# **Single-Cell Homeostatic Simulations**

# **Yu and Fyon Models — Average Neuron**

This notebook investigates homeostatic compensatory mechanisms 
in a single average dopaminergic neuron following targeted 
pharmacological blockades. Two conductance-based models are 
compared in parallel:
- **Yu model** (Yu & Canavier, 2015) — classical reference
- **Fyon model** (Fyon et al., 2025) — incorporates gPace (IXG)

**Simulation protocol:**
- Baseline phase : 0 → 30 s (no blockade)
- Blockade applied instantaneously at t = 30 s
- Homeostatic compensation observed until t = 300 s

**Blockades simulated:**
1. Baseline validation (no blockade)
2. Shift in half-activation voltages (Yu model only)
3. L-type calcium channel blockade (Cav1.3)
4. gPace conductance blockade
5. Augmented models with N-type and T-type channels
6. Fixed calcium leak + augmented models

# **Useful packages and functions**

In [ ]:
using DifferentialEquations, Plots, Polynomials, LaTeXStrings, ColorSchemes, DelimitedFiles, DataFrames
using Statistics, StatsPlots, Random, ProgressMeter, Printf, LinearAlgebra, Plots.PlotMeasures
include("DA_kinetics.jl") # Loading of DA kinetics of gating variables
include("Essaie.jl") # Loading of DA model
include("DA_utils.jl"); # Loading of some utils functions

# **Global parameters**

In [ ]:
# Definition of simulation time (in ms)
const Tfinal = 20000
const tspan  = (0.0, Tfinal)
tt = 0. : 0.01 : Tfinal
tt_rand = 0. : 1 : Tfinal

# Definition of reversal potential values (in mV), [Mg] and membrane capacitance
const VNa     = 60. # Sodium reversal potential
const VK      = -90. # Potassium reversal potential
const VCa     = 50. # Calcium reversal potential
const VH      = -29. # H reversal potential
const VLNS    = -65. # Leak reversal potential
const EPacemaker = 4.2732015978991615 # Reversal potential of pacemaking channels

const C       = 1. # Membrane capacitance
const fCa     = 0.018 # Fraction of unbuffered free calcium
const ICapmax = 11 # Maximum calcium pump current
const F       = 96520 # Faraday constant in ms*µA/mmol (and taking cm³=mL)
const d       = 15 # Soma diameter in cm
const L       = 25 # Soma length

# Definition of voltage range for the DICs
const Vmin = -100 
const Vmax = 50
const Vrange = range(Vmin, stop=Vmax, step=0.0154640);

## Plotting Configuration

In [ ]:
# Modifying backend GR attributes
gr(guidefontsize=25, tickfontsize=15, legendfontsize=12, margin=10Plots.mm, grid=false)
myApple = RGBA(187/255, 206/255, 131/255, 1)
mySalmon = RGBA(243/255, 124/255, 130/255)
myYellow = RGBA(228/255, 205/255, 121/255, 1)
myBlue = RGBA(131/255, 174/255, 218/255, 1)
myDarkBlue = RGBA(114/255, 119/255, 217/255, 1)
myOrange = RGBA(241/255, 175/255, 113/255, 1)
myPink = RGBA(243/255, 124/255, 130/255, 1)
myPurple = RGBA(169/255, 90/255, 179/255, 1)
myGreen = RGBA(132/255, 195/255, 168/255, 1)
myRed = RGBA(158/255, 3/255, 8/255, 1)
myGray = RGBA(150/255, 150/255, 150/255, 1)
myLightBlue = RGBA(127/255, 154/255, 209/255, 1);
default(fmt = :png);

In [ ]:
# Define a struct (optional, but useful if you need parameters)
struct NoisyFunction
    amplitude::Float64  # amplitude of the noise
end

# Overload the () operator to make the struct callable
function (nf::NoisyFunction)(x::Float64)
    noise = nf.amplitude * randn()  # Generate Gaussian noise (mean 0, std 1)
    return noise  # Example function with noise
end

function condition(u,t,integrator) # Event when event_f(u,t) == 0
  (u[1]- (-20.))
end

function affect!(integrator)
end

cb = ContinuousCallback(condition, affect!, nothing, save_positions = (true, false));

# **Simulations**

# **Baseline Validation — No Blockade for Yu Model**
**Objective:** Verify that the homeostatic controller does not 
interfere with baseline pacemaking activity for the YU and Fyon model

In [ ]:
moving_average(vs, n, padding) = [sum(vs[i:(i+n-1)])/n for i in 1:padding:(length(vs)-(n-1))];

In [ ]:
#Définition des paramètres du controleur homeostatique
Ca_tgt = 0.00015   
tau_g  = 0.01 
t_Na  = 0.06

gNa = 6.0
gCaL =0.139
gKd = 1.117
gKA = 1.68
gKERG = 0.13
gKSK = 0.07
gH = 0.078
gLNS = 0.28
gLCa = 0.00245

tCaL = t_Na * gNa / gCaL
tKd = t_Na * gNa/gKd 
tKA =  t_Na * gNa/ gKA
tKERG =  t_Na * gNa/ gKERG
tKSK =  t_Na * gNa/gKSK
tH =  t_Na * gNa/gH
tLNS =  t_Na * gNa/gLNS
tLCa =  t_Na * gNa/gLCa

Iapp_func(t) = 0.0 

p_homeo = (Iapp_func, Ca_tgt, tau_g,  t_Na, tCaL, tKd, tKA, tKERG, tKSK, tH, tLNS, tLCa)

V0 = 0.0
Ca0 = 2e-4 

u_biophys = [
    V0, m_inf(V0), h_inf(V0), hs_inf(V0), l_inf(V0), n_inf(V0), 
    p_inf(V0), q1_inf(V0), q2_inf(V0), 0.0, 0.0, mH_inf(V0), Ca0
]

g_init = [gNa, gCaL, gKd, gKA, gKERG, gKSK, gH, gLNS, gLCa]

m_init = copy(g_init)

u0_homeo = vcat(u_biophys, g_init, m_init)

T_long = 300000.0  
tspan_long = (0.0, T_long)

prob_2014 = ODEProblem(DA_homeo_2014_ODE, u0_homeo, tspan_long, p_homeo)

sol_2014 = solve(prob_2014, 
    RadauIIA5(),             
    reltol=1e-5,            
    abstol=1e-10,          
    maxiters=1e8, 
    isoutofdomain = (u,p,t) -> any(x -> x < 0, u[13:31]), 
    dtmax=1.0)             

if sol_2014.t[end] < T_long
    @warn "Crash persistant à t = $(sol_2014.t[end])"
end


# --- VISULAISATION TRACE ELECTRIQUE ---
tt_long = 0.0 : 10.0 : sol_2014.t[end]
res_2014 = sol_2014(tt_long)

t_zoom1 = 5000 : 0.1 : 7000
res_zoom1 = sol_2014(t_zoom1)
p_v1 = plot(t_zoom1 ./ 1e3, res_zoom1[1, :], linewidth=1.5, color=myDarkBlue, 
   legend=false, ylims=(-90, 30), xlims=(5, 7), 
    title="Electrical Trace 5-7 sec ",
    xticks=([5, 6, 7], ["5", "6", "7"]))
ylabel!("V (mV)")

t_zoom = 291000 : 0.1 : 293000 
res_zoom = sol_2014(t_zoom) 

p_v_2014 = plot(t_zoom ./ 1e3, res_zoom[1, :], linewidth=1.5, color=myDarkBlue, 
    legend=false, ylims=(-90, 30), xlims=(291, 293), 
    title="Electrical Trace 291-293 sec",
    xticks=([291, 292, 293], ["291", "292", "293"]))
ylabel!("V (mV)")


# --- VISUALISATION DU CALCIUM ---
ca_trace = res_2014[13, :]
ca_avg = moving_average(ca_trace, 300, 1) 
t_avg = range(0, 300, length=length(ca_avg))

p_ca = plot(t_avg, ca_avg, linewidth=2, color=:black, label="[Ca] moyen", title="Calcium convergence")
hline!([Ca_tgt], color=:red, linestyle=:dash, label="Cible (Ca_tgt)")
ylabel!("[Ca]")


# --- VISUALISATION DES CONDUCTANCES ---
names_2014 = ["gNa", "gCaL", "gKd", "gKA", "gKERG", "gKSK", "gH", "gLNS", "gLCa"]
p_gs = plot(legend=:outerright, title="Conductance evolution", yaxis=:log) # 

for i in 1:9
    plot!(tt_long ./ 1000, res_2014[13+i, :], label=names_2014[i])
end
ylabel!("g (mS/cm²)")
xlabel!("Time (s)")

# --- AFFICHAGE FINAL ---
l = @layout [a{0.25h}; b{0.25h}; c{0.25h}; d{0.25h} ]
plot(p_v1,p_v_2014, p_ca, p_gs, layout=l, size=(1000, 1000), margin=10Plots.px) 

# **Baseline Validation — No Blockade for Fyon Model**

In [ ]:
#Définition des paramètres du controleur homeostatique
Ca_tgt = 7.81e-5   
tau_g  = 0.01     
t_Na  = 0.06    


gNa = 25.0
gCaL = 1.0
gKd = 10.0
gKA = 1.68
gKERG = 0.13
gKSK = 0.3
gH = 0.078
gPace = 5
gLNS = 0.01   
gLCa = 0.00245 

tCaL = t_Na * gNa/ gCaL
tKd = t_Na * gNa/gKd 
tKA =  t_Na * gNa/ gKA
tKERG =  t_Na * gNa/ gKERG
tKSK =  t_Na * gNa/gKSK
tH =  t_Na * gNa/gH
tLNS =  t_Na * gNa/gLNS
tLCa =  t_Na * gNa/gLCa
tPace = t_Na * gNa/gPace

Iapp_func(t) = 0.0  
     
p_homeo = (Iapp_func, Ca_tgt, tau_g, t_Na, tCaL, tKd, tKA, tKERG, tKSK, tH, tPace, tLNS, tLCa)

V0 = -50.0
Ca0 = 2e-4

u_biophys = [
    V0, m_inf_true(V0), h_inf(V0), hs_inf(V0), l_inf_true(V0), n_inf(V0), 
    p_inf(V0), q1_inf(V0), q2_inf(V0), 0.0, 0.0, mH_inf(V0), Ca0
]

g_main_init = [gNa, gCaL, gKd, gKA, gKERG, gKSK, gH,gPace]

m_main_init = copy(g_main_init)

g_leak_init = [gLNS,  gLCa]

m_leak_init = copy(g_leak_init)

u0_homeo = vcat(u_biophys, g_main_init, m_main_init, g_leak_init, m_leak_init)

T_long = 300000.0 
tspan_long = (0.0, T_long)

prob_rigorous = ODEProblem(DA_homeo_ODE, u0_homeo, tspan_long, p_homeo)

sol_rigorous = solve(prob_rigorous, 
    KenCarp47(),            
    reltol=1e-5,            
    abstol=1e-8,           
    maxiters=1e9, 
    isoutofdomain = (u,p,t) -> any(x -> x < 0, u[13:33]), 
    dtmax=1.0)               

if sol_rigorous.t[end] < T_long
    @warn "Crash persistant à t = $(sol_rigorous.t[end])."
end

# --- VISUALISATION TRACE ELECTRIQUE---
tt_long = 0.0 : 10.0 : sol_rigorous.t[end]
res_rigorous = sol_rigorous(tt_long)

t_zoom = 291000 : 0.1 : 293000 
res_zoom = sol_rigorous(t_zoom) 

p_v_rig = plot(t_zoom ./ 1e3, res_zoom[1, :], linewidth=1.5, color=myDarkBlue, 
    legend=false, ylims=(-90, 30), xlims=(291, 293), 
    title="Electrical Trace 291-293 sec",
    xticks=([291, 292, 293], ["292", "292", "293"]))
ylabel!("V (mV)")

# --- VISUALISATION DU CALCIUM ---
ca_trace_rig = res_rigorous[13, :]
ca_avg_rig = moving_average(ca_trace_rig, 300, 1)
t_avg_rig = range(0, 300, length=length(ca_avg_rig))

p_ca_rig = plot(t_avg_rig, ca_avg_rig, linewidth=2, color=:black, label="[Ca] moyen", title="Calcium convergence")
hline!([Ca_tgt], color=:red, linestyle=:dash, linewidth=4, label="Ca_tgt")
ylabel!("[Ca]")

# --- VISUALISATION DES CONDUCTANCES ---
p_gs_rig = plot(legend=:outerright, title="Conductance evolution", yaxis=:log)

main_names = ["gNa", "gCaL", "gKd", "gKA", "gKERG", "gKSK", "gH", "gPace"]
for i in 1:8
    plot!(tt_long ./ 1000, res_rigorous[13+i, :], label=main_names[i])
end
# Fuites dynamiques aux indices 30 et 31 
plot!(tt_long ./ 1000, res_rigorous[30, :], label="gLNS")
plot!(tt_long ./ 1000, res_rigorous[31, :], label="gLCa")

ylabel!("g (mS/cm²)")
xlabel!("Time (s)")

# --- AFFICHAGE FINAL ---
l = @layout [a{0.33h}; b{0.33h}; c{0.34h}]
plot(p_v_rig, p_ca_rig, p_gs_rig, layout=l, size=(1000, 1000), margin=10Plots.px)


# **Shift in Half-Activation Voltages (Yu model only)**
**Objective:** Test whether the homeostatic controller can 
restore pacemaking after shifting sodium and L-type calcium 
channel activation to physiological values (−10 mV). 

In [ ]:
#Définition des paramètres du controleur homeostatique
Ca_tgt = 0.00015   
tau_g  = 0.01 
t_Na  = 0.06

gNa = 6.0
gCaL =0.139
gKd = 1.117
gKA = 1.68
gKERG = 0.13
gKSK = 0.07
gH = 0.078
gLNS = 0.28
gLCa = 0.00245

tCaL = t_Na * gNa / gCaL
tKd = t_Na * gNa/gKd 
tKA =  t_Na * gNa/ gKA
tKERG =  t_Na * gNa/ gKERG
tKSK =  t_Na * gNa/gKSK
tH =  t_Na * gNa/gH
tLNS =  t_Na * gNa/gLNS
tLCa =  t_Na * gNa/gLCa

Iapp_func(t) = 0.0 

p_homeo = (Iapp_func, Ca_tgt, tau_g,  t_Na, tCaL, tKd, tKA, tKERG, tKSK, tH, tLNS, tLCa)

V0 = 0.0
Ca0 = 2e-4 

u_biophys = [
    V0, m_inf(V0), h_inf(V0), hs_inf(V0), l_inf(V0), n_inf(V0), 
    p_inf(V0), q1_inf(V0), q2_inf(V0), 0.0, 0.0, mH_inf(V0), Ca0
]

g_init = [gNa, gCaL, gKd, gKA, gKERG, gKSK, gH, gLNS, gLCa]

m_init = copy(g_init)

u0_homeo = vcat(u_biophys, g_init, m_init)

T_long = 300000.0  
tspan_long = (0.0, T_long)

prob_2014 = ODEProblem(DA_homeo_2014_ODE_linf, u0_homeo, tspan_long, p_homeo)

sol_2014 = solve(prob_2014, 
    RadauIIA5(),             
    reltol=1e-5,            
    abstol=1e-10,          
    maxiters=1e8, 
    isoutofdomain = (u,p,t) -> any(x -> x < 0, u[13:31]), 
    dtmax=1.0)             

if sol_2014.t[end] < T_long
    @warn "Crash persistant à t = $(sol_2014.t[end])"
end


# --- VISULAISATION TRACE ELECTRIQUE ---
tt_long = 0.0 : 10.0 : sol_2014.t[end]
res_2014 = sol_2014(tt_long)

t_zoom1 = 5000 : 0.1 : 7000
res_zoom1 = sol_2014(t_zoom1)
p_v1 = plot(t_zoom1 ./ 1e3, res_zoom1[1, :], linewidth=1.5, color=myDarkBlue, 
   legend=false, ylims=(-90, 30), xlims=(5, 7), 
    title="Electrical Trace 5-7 sec ",
    xticks=([5, 6, 7], ["5", "6", "7"]))
ylabel!("V (mV)")

t_zoom = 291000 : 0.1 : 293000 
res_zoom = sol_2014(t_zoom) 

p_v_2014 = plot(t_zoom ./ 1e3, res_zoom[1, :], linewidth=1.5, color=myDarkBlue, 
    legend=false, ylims=(-90, 30), xlims=(291, 293), 
    title="Electrical Trace 291-293 sec",
    xticks=([291, 292, 293], ["291", "292", "293"]))
ylabel!("V (mV)")


# --- VISUALISATION DU CALCIUM ---
ca_trace = res_2014[13, :]
ca_avg = moving_average(ca_trace, 300, 1) 
t_avg = range(0, 300, length=length(ca_avg))

p_ca = plot(t_avg, ca_avg, linewidth=2, color=:black, label="[Ca] moyen", title="Calcium convergence")
hline!([Ca_tgt], color=:red, linestyle=:dash, label="Cible (Ca_tgt)")
ylabel!("[Ca]")


# --- VISUALISATION DES CONDUCTANCES ---
names_2014 = ["gNa", "gCaL", "gKd", "gKA", "gKERG", "gKSK", "gH", "gLNS", "gLCa"]
p_gs = plot(legend=:outerright, title="Conductance evolution", yaxis=:log) # 

for i in 1:9
    plot!(tt_long ./ 1000, res_2014[13+i, :], label=names_2014[i])
end
ylabel!("g (mS/cm²)")
xlabel!("Time (s)")

# --- AFFICHAGE FINAL ---
l = @layout [a{0.25h}; b{0.25h}; c{0.25h}; d{0.25h} ]
plot(p_v1,p_v_2014, p_ca, p_gs, layout=l, size=(1000, 1000), margin=10Plots.px) 

# **L-type Calcium Channel Blockade for Yu Model (Cav1.3)**

In [ ]:
#Définition des paramètres du controleur homeostatique
Ca_tgt = 0.00015    
tau_g  = 0.01
t_Na  = 0.06 

gNa = 6.0
gCaL =0.139
gKd = 1.117
gKA = 1.68
gKERG = 0.13
gKSK = 0.07
gH = 0.078
gLNS = 0.28
gLCa = 0.00245

tCaL = t_Na * gNa / gCaL
tKd = t_Na * gNa/gKd 
tKA =  t_Na * gNa/ gKA
tKERG =  t_Na * gNa/ gKERG
tKSK =  t_Na * gNa/gKSK
tH =  t_Na * gNa/gH
tLNS =  t_Na * gNa/gLNS
tLCa =  t_Na * gNa/gLCa

Iapp_func(t) = 0.0  # pA

p_homeo = (Iapp_func, Ca_tgt, tau_g,  t_Na, tCaL, tKd, tKA, tKERG, tKSK, tH, tLNS, tLCa)

V0 = 0.0
Ca0 = 1e-4

u_biophys = [
    V0, m_inf(V0), h_inf(V0), hs_inf(V0), l_inf(V0), n_inf(V0), 
    p_inf(V0), q1_inf(V0), q2_inf(V0), 0.0, 0.0, mH_inf(V0), Ca0
]

g_init = [gNa, gCaL, gKd, gKA, gKERG, gKSK, gH, gLNS, gLCa]

m_init = copy(g_init)

u0_homeo = vcat(u_biophys, g_init, m_init)

T_long = 300000.0  
tspan_long = (0.0, T_long)

prob_2014 = ODEProblem( DA_homeo_2014_ODE_blockL, u0_homeo, tspan_long, p_homeo)

sol_2014 = solve(prob_2014, 
    RadauIIA5(),             
    reltol=1e-5,             
    abstol=1e-10,            
    maxiters=1e8, 
    isoutofdomain = (u,p,t) -> any(x -> x < 0, u[13:31]),
    dtmax=1.0)               

if sol_2014.t[end] < T_long
    @warn "Crash persistant à t = $(sol_2014.t[end])."
end

# --- VISUALISATION TRACE ELECTRIQUE ---
tt_long = 0.0 : 10.0 : sol_2014.t[end]
res_2014 = sol_2014(tt_long)

t_zoom1 = 5000 : 0.1 : 7000
res_zoom1 = sol_2014(t_zoom1)
p_v1 = plot(t_zoom1 ./ 1e3, res_zoom1[1, :], linewidth=1.5, color=myDarkBlue, 
    legend=false, ylims=(-90, 30), xlims=(5, 7), 
    title="Electrical trace 5-7 sec",
    xticks=([5, 6, 7], ["5", "6", "7"]))
ylabel!("V (mV)")

t_zoom = 291000 : 0.1 : 293000 
res_zoom = sol_2014(t_zoom) 

p_v_2014 = plot(t_zoom ./ 1e3, res_zoom[1, :], linewidth=1.5, color=myDarkBlue, 
    legend=false, ylims=(-90, 30), xlims=(291, 293), 
    title="Electrical trace 291-293 sec",
    xticks=([291, 292, 293], ["291", "292", "293"]))
ylabel!("V (mV)")


# --- VISUALISATION DU CALCIUM ---
ca_trace = res_2014[13, :]
ca_avg = moving_average(ca_trace, 300, 1) 
t_avg = range(0, 300, length=length(ca_avg))

p_ca = plot(t_avg, ca_avg, linewidth=2, color=:black, label="[Ca] moyen", title="Calcium convergence ")
hline!([Ca_tgt], color=:red, linestyle=:dash, label="Cible (Ca_tgt)")
ylabel!("[Ca]")


# --- VISUALISATION DES CONDUCTANCES ---
p_gs = plot(legend=:outerright, title="Conductance evolution", yaxis=:log)

gCaL_eff = [ (t < 30.0) ? res_2014[15, i] : 1e-18 for (i, t) in enumerate(tt_long ./ 1000) ]

names_2014 = ["gNa", "gCaL", "gKd", "gKA", "gKERG", "gKSK", "gH", "gLNS", "gLCa"]

for i in [1, 3, 4, 5, 6, 7, 8, 9] # On saute i=2 (gCaL)
    plot!(tt_long ./ 1000, res_2014[13+i, :], label=names_2014[i])
end

plot!(tt_long ./ 1000, gCaL_eff, label="gCaL", color=:red, linewidth=2)
ylims!(1e-6, 1e4)

ylabel!("g (mS/cm²)")
xlabel!("Time (s)")

# --- AFFICHAGE FINAL ---
l = @layout [a{0.25h}; b{0.25h}; c{0.25h}; d{0.25h}]
plot(p_v1, p_v_2014, p_ca, p_gs, layout=l, size=(1000, 1000), margin=10Plots.px) 

# **L-type Calcium Channel Blockade (Cav1.3) for Fyon Model**

In [ ]:
#Définition des paramètres du controleur homeostatique
Ca_tgt = 7.81e-5    
tau_g  = 0.01   
t_Na  = 0.06  


gNa = 25.0
gCaL = 1.0
gKd = 10.0
gKA = 1.68
gKERG = 0.13
gKSK = 0.3
gH = 0.078
gPace = 5
gLNS = 0.01   
gLCa = 0.00245 

tCaL = t_Na * gNa/ gCaL
tKd = t_Na * gNa/gKd 
tKA =  t_Na * gNa/ gKA
tKERG =  t_Na * gNa/ gKERG
tKSK =  t_Na * gNa/gKSK
tH =  t_Na * gNa/gH
tLNS =  t_Na * gNa/gLNS
tLCa =  t_Na * gNa/gLCa
tPace = t_Na * gNa/gPace

Iapp_func(t) = 0.0  
     
p_homeo = (Iapp_func, Ca_tgt, tau_g, t_Na, tCaL, tKd, tKA, tKERG, tKSK, tH, tPace, tLNS, tLCa)

V0 = -50.0
Ca0 = 1e-4

u_biophys = [
    V0, m_inf_true(V0), h_inf(V0), hs_inf(V0), l_inf_true(V0), n_inf(V0), 
    p_inf(V0), q1_inf(V0), q2_inf(V0), 0.0, 0.0, mH_inf(V0), Ca0
]

g_main_init = [gNa, gCaL, gKd, gKA, gKERG, gKSK, gH,gPace]

m_main_init = copy(g_main_init)

g_leak_init = [gLNS,  gLCa]

m_leak_init = copy(g_leak_init)

u0_homeo = vcat(u_biophys, g_main_init, m_main_init, g_leak_init, m_leak_init)

T_long = 300000.0 
tspan_long = (0.0, T_long)

prob_rigorous = ODEProblem(DA_homeo_ODE_blockL, u0_homeo, tspan_long, p_homeo)

sol_rigorous = solve(prob_rigorous, 
    KenCarp47(),             
    reltol=1e-5,             
    abstol=1e-8,            
    maxiters=1e9, 
    isoutofdomain = (u,p,t) -> any(x -> x < 0, u[13:33]),
    dtmax=1.0)             

if sol_rigorous.t[end] < T_long
    @warn "Crash persistant à t = $(sol_rigorous.t[end])."
end

# --- VISUALISATION TRACE ELECTRIQUE ---
tt_long = 0.0 : 10.0 : sol_rigorous.t[end]
res_rigorous = sol_rigorous(tt_long)

t_zoom1 = 5000 : 0.1 : 7000
res_zoom1 = sol_rigorous(t_zoom1)
p_v1 = plot(t_zoom1 ./ 1e3, res_zoom1[1, :], linewidth=1.5, color=myDarkBlue, 
    legend=false, ylims=(-90, 30), xlims=(5, 7), 
    title="Electrical Trace 5-7 sec ",
    xticks=([5, 6, 7], ["5", "6", "7"]))
ylabel!("V (mV)")

t_zoom = 291000 : 0.1 : 293000 
res_zoom = sol_rigorous(t_zoom) 

p_v_rig = plot(t_zoom ./ 1e3, res_zoom[1, :], linewidth=1.5, color=myDarkBlue, 
    legend=false, ylims=(-90, 30), xlims=(291, 293), 
    title="Electrical Trace 291-293 sec",
    xticks=([291, 292, 293], ["291", "292", "293"]))
ylabel!("V (mV)")

# --- VISUALISATION DU CALCIUM ---
ca_trace_rig = res_rigorous[13, :]
ca_avg_rig = moving_average(ca_trace_rig, 300, 1)
t_avg_rig = range(0, 300, length=length(ca_avg_rig))

p_ca_rig = plot(t_avg_rig, ca_avg_rig, linewidth=2, color=:black, label="[Ca] moyen", title="Calcium convergence")
hline!([Ca_tgt], color=:red, linestyle=:dash, label="Ca_tgt")
ylabel!("[Ca]")

# --- VISUALISATION DES CONDUCTANCES  ---
p_gs_rig = plot(legend=:outerright, title="Conductance evolution", yaxis=:log)

ts_sec = tt_long ./ 1000

main_names = ["gNa", "gCaL", "gKd", "gKA", "gKERG", "gKSK", "gH", "gPace"]

for i in [1, 3, 4, 5, 6, 7, 8]
    plot!(ts_sec, res_rigorous[13+i, :], label=main_names[i])
end

gCaL_eff = [ (t < 30.0) ? res_rigorous[15, i] : 1e-18 for (i, t) in enumerate(ts_sec) ]
plot!(ts_sec, gCaL_eff, label="gCaL", 
      color=:red, linewidth=2.5)

plot!(ts_sec, res_rigorous[30, :], label="gLNS", alpha=0.7)
plot!(ts_sec, res_rigorous[31, :], label="gLCa", alpha=0.7)
ylims!(1e-6, 1e4)
ylabel!("g (mS/cm²)")
xlabel!("Time (s)")

# --- AFFICHAGE FINAL ---
l = @layout [a{0.25h}; b{0.25h}; c{0.25h}; d{0.25h}]
plot(p_v1,p_v_rig, p_ca_rig,p_gs_rig, layout=l, size=(1000, 1000), margin=10Plots.px)

# **gPace Conductance Blockade for Fyon Model**

In [ ]:
#Définition des paramètres du controleur homeostatique

#REVOIR
Ca_tgt = 7.81e-5    
tau_g  = 0.01     
t_Na  = 0.06    

gNa = 25.0
gCaL = 1.0
gKd = 10.0
gKA = 1.68
gKERG = 0.13
gKSK = 0.3
gH = 0.078
gPace = 5
gLNS = 0.01   
gLCa = 0.00245 

tCaL = t_Na * gNa/ gCaL
tKd = t_Na * gNa/gKd 
tKA =  t_Na * gNa/ gKA
tKERG =  t_Na * gNa/ gKERG
tKSK =  t_Na * gNa/gKSK
tH =  t_Na * gNa/gH
tLNS =  t_Na * gNa/gLNS
tLCa =  t_Na * gNa/gLCa
tPace = t_Na * gNa/gPace

Iapp_func(t) = 0.0  
     
p_homeo = (Iapp_func, Ca_tgt, tau_g, t_Na, tCaL, tKd, tKA, tKERG, tKSK, tH, tPace, tLNS, tLCa)

V0 = -50.0
Ca0 = 1e-4

u_biophys = [
    V0, m_inf_true(V0), h_inf(V0), hs_inf(V0), l_inf_true(V0), n_inf(V0), 
    p_inf(V0), q1_inf(V0), q2_inf(V0), 0.0, 0.0, mH_inf(V0), Ca0
]

g_main_init = [gNa, gCaL, gKd, gKA, gKERG, gKSK, gH,gPace]

m_main_init = copy(g_main_init)

g_leak_init = [gLNS,  gLCa]

m_leak_init = copy(g_leak_init)

u0_homeo = vcat(u_biophys, g_main_init, m_main_init, g_leak_init, m_leak_init)

T_long = 300000.0 
tspan_long = (0.0, T_long)

prob_rigorous = ODEProblem(DA_homeo_ODE_blockgpace, u0_homeo, tspan_long, p_homeo)

sol_rigorous = solve(prob_rigorous, 
    KenCarp47(),             
    reltol=1e-5,            
    abstol=1e-8,            
    maxiters=1e9, 
    isoutofdomain = (u,p,t) -> any(x -> x < 0, u[13:33]), 
    dtmax=1.0)              

if sol_rigorous.t[end] < T_long
    @warn "Crash persistant à t = $(sol_rigorous.t[end])."
end

# --- VISUALISATION TRACE ELECTRIQUE ---
tt_long = 0.0 : 10.0 : sol_rigorous.t[end]
res_rigorous = sol_rigorous(tt_long)

t_zoom1 = 5000 : 0.1 : 7000
res_zoom1 = sol_rigorous(t_zoom1)
p_v1 = plot(t_zoom1 ./ 1e3, res_zoom1[1, :], linewidth=1.5, color=myDarkBlue, 
    legend=false, ylims=(-90, 30), xlims=(5, 7), 
    title="Electrical Trace 5-7 sec ",
    xticks=([5, 6, 7], ["5", "6", "7"]))
ylabel!("V (mV)")

t_zoom = 291000 : 0.1 : 293000 
res_zoom = sol_rigorous(t_zoom) 

p_v_rig = plot(t_zoom ./ 1e3, res_zoom[1, :], linewidth=1.5, color=myDarkBlue, 
    legend=false, ylims=(-90, 30), xlims=(291, 293), 
    title="Electrical Trace 291-293 sec",
    xticks=([291, 292, 293], ["291", "292", "293"]))
ylabel!("V (mV)")

# --- VISUALISATION DU CALCIUM ---
ca_trace_rig = res_rigorous[13, :]
ca_avg_rig = moving_average(ca_trace_rig, 300, 1)
t_avg_rig = range(0, 300, length=length(ca_avg_rig))

p_ca_rig = plot(t_avg_rig, ca_avg_rig, linewidth=2, color=:black, label="[Ca] moyen", title="Calcium Convergence")
hline!([Ca_tgt], color=:red, linestyle=:dash, label="Ca_tgt")
ylabel!("[Ca]")

# --- VISUALISATION DES CONDUCTANCES  ---
p_gs_rig = plot(legend=:outerright, title="Conductance Evolution", yaxis=:log)

ts_sec = tt_long ./ 1000

main_names = ["gNa", "gCaL", "gKd", "gKA", "gKERG", "gKSK", "gH", "gPace"]

for i in [1, 2, 3, 4, 5, 6, 7] # On exclut l'indice 2 (gCaL) ici
    plot!(ts_sec, res_rigorous[13+i, :], label=main_names[i])
end

gPace_eff = [ (t < 30.0) ? res_rigorous[21, i] : 1e-18 for (i, t) in enumerate(ts_sec) ]
plot!(ts_sec, gPace_eff, label="gPace", 
      color=:red, linewidth=2.5)

plot!(ts_sec, res_rigorous[30, :], label="gLNS", alpha=0.7)
plot!(ts_sec, res_rigorous[31, :], label="gLCa", alpha=0.7)
ylims!(1e-6, 1e4)
ylabel!("g (mS/cm²)")
xlabel!("Time (s)")

# --- AFFICHAGE FINAL ---
l = @layout [a{0.25h}; b{0.25h}; c{0.25h} ; d{0.25h}]
plot(p_v1,p_v_rig, p_ca_rig, p_gs_rig, layout=l, size=(1000, 1000), margin=10Plots.px)

# **Augmented Models — N-type and T-type Channels Added for YU Model**
# **Baseline Simulation**
Test whether alternative calcium sources 
(Cav2.2, Cav3.1) influence homeostatic compensation after 
L-type blockade.  

In [ ]:
#Définition des paramètres du controleur homeostatique
Ca_tgt = 0.00015    
tau_g  = 0.01
t_Na  = 0.06 

gNa = 6.0
gCaL =0.139
gKd = 1.117
gKA = 1.68
gKERG = 0.13
gKSK = 0.07
gH = 0.078
gLNS = 0.28
gLCa = 0.00245
gNtype = 0.2
gTtype = 0.025

tCaL = t_Na * gNa / gCaL
tKd = t_Na * gNa/gKd 
tKA =  t_Na * gNa/ gKA
tKERG =  t_Na * gNa/ gKERG
tKSK =  t_Na * gNa/gKSK
tH =  t_Na * gNa/gH
tLNS =  t_Na * gNa/gLNS
tLCa =  t_Na * gNa/gLCa
tNCa = t_Na* gNa/ gNtype
tTCA = t_Na* gNa/ gTtype

Iapp_func(t) = 0.0  
p_homeo = (Iapp_func, Ca_tgt, tau_g,  t_Na, tCaL, tKd, tKA, tKERG, tKSK, tH, tLNS, tLCa, tNCa, tTCA)

V0 = 0.0
Ca0 = 2e-4 

u_biophys = [
    V0, m_inf(V0), h_inf(V0), hs_inf(V0), l_inf(V0), n_inf(V0), 
    p_inf(V0), q1_inf(V0), q2_inf(V0), 0.0, 0.0, mH_inf(V0), Ca0
]

g_init = [gNa, gCaL, gKd, gKA, gKERG, gKSK, gH, gLNS, gLCa]

m_init = copy(g_init)

g_cal = [gNtype, gTtype]

m_cal = copy(g_cal)

u_phys = [mN_inf(V0), hN_inf(V0), mT_inf(V0), hT_inf(V0)]

u0_homeo = vcat(u_biophys, g_init, m_init, g_cal, m_cal, u_phys)

T_long = 300000.0  
tspan_long = (0.0, T_long)

prob_2014 = ODEProblem(DA_homeo_2014_ODE_Ntype_Ttype, u0_homeo, tspan_long, p_homeo)

sol_2014 = solve(prob_2014, 
    RadauIIA5(),             
    reltol=1e-5,             
    abstol=1e-10,            
    maxiters=1e8, 
    isoutofdomain = (u,p,t) -> any(x -> x < 0, u[13:35]), 
    dtmax=1.0)               

if sol_2014.t[end] < T_long
    @warn "Crash persistant à t = $(sol_2014.t[end])."
end

# --- VISUALISATION TRACE ELECTRIQUE ---
tt_long = 0.0 : 10.0 : sol_2014.t[end]
res_2014 = sol_2014(tt_long)

t_zoom = 291000 : 0.1 : 293000 
res_zoom = sol_2014(t_zoom) 

p_v_2014 = plot(t_zoom ./ 1e3, res_zoom[1, :], linewidth=1.5, color=myDarkBlue, 
    legend=false, ylims=(-90, 30), xlims=(291, 293), 
    title="Electrical Trace 291-293 sec",
    xticks=([291, 292, 293], ["291", "292", "293"]))
ylabel!("V (mV)")


# --- VISUALISATION DU CALCIUM ---
ca_trace = res_2014[13, :]
ca_avg = moving_average(ca_trace, 300, 1) 
t_avg = range(0, 300, length=length(ca_avg))

p_ca = plot(t_avg, ca_avg, linewidth=2, color=:black, label="[Ca] moyen", title="Calcium Convergence")
hline!([Ca_tgt], color=:red, linestyle=:dash, linewidth=4, label="Ca_tgt")
ylabel!("[Ca]")

# --- VISUALISATION DES CONDUCTANCES ---
names_2014 = ["gNa", "gCaL", "gKd", "gKA", "gKERG", "gKSK", "gH", "gLNS", "gLCa"]
p_gs = plot(legend=:outerright, title="Conductance Evolution", yaxis=:log) # 

for i in 1:9
    plot!(tt_long ./ 1000, res_2014[13+i, :], label=names_2014[i])
end
plot!(tt_long ./ 1000, res_2014[32, :], label="gNtype", linewidth=2, linestyle=:dash)
plot!(tt_long ./ 1000, res_2014[33, :], label="gTtype", linewidth=2, linestyle=:dash)

ylabel!("g (mS/cm²)")
xlabel!("Time (s)")

# --- AFFICHAGE FINAL ---
l = @layout [a{0.25h}; b{0.25h}; c{0.5h}]
pt = plot(p_v_2014, p_ca, p_gs, layout=l, size=(1000, 1000), margin=10Plots.px) 
display(pt)

# **Augmented Models — N-type and T-type Channels Added for Fyon Model**
# **Baseline Simulation**
Test whether alternative calcium sources 
(Cav2.2, Cav3.1) influence homeostatic compensation after 
L-type blockade. 

In [ ]:
#Définition des paramètres du controleur homeostatique
Ca_tgt = 7.8474e-5   
tau_g  = 0.01     
t_Na  = 0.06    

gNa = 25
gCaL = 1.0
gKd = 10.0
gKA = 1.68
gKERG = 0.13
gKSK = 0.3
gH = 0.078
gPace = 5
gLNS = 0.01   
gLCa = 0.00245 
gNtype = 0.2
gTtype = 0.025

tCaL = t_Na * gNa/ gCaL
tKd = t_Na * gNa/gKd 
tKA =  t_Na * gNa/ gKA
tKERG =  t_Na * gNa/ gKERG
tKSK =  t_Na * gNa/gKSK
tH =  t_Na * gNa/gH
tLNS =  t_Na * gNa/gLNS
tLCa =  t_Na * gNa/gLCa
tPace = t_Na * gNa/gPace
tNCa = t_Na * gNa/gNtype
tTCA = t_Na * gNa/gTtype

Iapp_func(t) = 0.0  
  
p_homeo = (Iapp_func, Ca_tgt, tau_g, t_Na, tCaL, tKd, tKA, tKERG, tKSK, tH, tPace, tLNS, tLCa, tNCa, tTCA)

V0 = -50
Ca0 = 2e-4

u_biophys = [
    V0, m_inf_true(V0), h_inf(V0), hs_inf(V0), l_inf_true(V0), n_inf(V0), 
    p_inf(V0), q1_inf(V0), q2_inf(V0), 0.0, 0.0, mH_inf(V0), Ca0
]

g_main_init = [gNa, gCaL, gKd, gKA, gKERG, gKSK, gH,gPace]

m_main_init = copy(g_main_init)

g_leak_init = [gLNS,  gLCa]

m_leak_init = copy(g_leak_init)

g_calcique = [gNtype, gTtype]

m_calcique = copy(g_calcique)

uphys = [mN_inf(V0), hN_inf(V0), mT_inf(V0), hT_inf(V0)]

u0_homeo = vcat(u_biophys, g_main_init, m_main_init, g_leak_init, m_leak_init, g_calcique, m_calcique, uphys)

T_long = 300000.0 
tspan_long = (0.0, T_long)

prob_rigorous = ODEProblem(DA_homeo_ODE_Ntype_Ttype_gpace, u0_homeo, tspan_long, p_homeo)

sol_rigorous = solve(prob_rigorous, 
    KenCarp47(),             
    reltol=1e-5,            
    abstol=1e-8,           
    maxiters=1e9, 
    isoutofdomain = (u,p,t) -> any(x -> x < 0, u[13:37]), 
    dtmax=1.0)               
if sol_rigorous.t[end] < T_long
    @warn "Crash persistant à t = $(sol_rigorous.t[end])"
end


# --- VISUALISATION TRACE ELECTRIQUE ---

tt_long = 0.0 : 10.0 : sol_rigorous.t[end]
res_rigorous = sol_rigorous(tt_long)

t_zoom = 291000 : 0.1 : 293000 
res_zoom = sol_rigorous(t_zoom) 

p_v_rig = plot(t_zoom ./ 1e3, res_zoom[1, :], linewidth=1.5, color=myDarkBlue, 
    legend=false, ylims=(-90, 30), xlims=(291, 293), 
    title="Electrical Trace 291-293 sec",
    xticks=([291, 292, 293], ["291", "292", "293"]))
ylabel!("V (mV)")

# --- VISUALISATION DU CALCIUM ---
ca_trace_rig = res_rigorous[13, :]
ca_avg_rig = moving_average(ca_trace_rig, 300, 1)
t_avg_rig = range(0, 300, length=length(ca_avg_rig))

p_ca_rig = plot(t_avg_rig, ca_avg_rig, linewidth=2, color=:black, label="[Ca] moyen", title="Calcium Convergence")
#hline!([Ca_tgt], color=:red, linestyle=:dash, label="Ca_tgt")
hline!([Ca_tgt], color=:red, linestyle=:dash, linewidth=4, label="Ca_tgt")
ylabel!("[Ca]")

# --- VISUALISATION DES CONDUCTANCES ---
p_gs_rig = plot(legend=:outerright, title="Conductance Evolution", yaxis=:log)

main_names = ["gNa", "gCaL", "gKd", "gKA", "gKERG", "gKSK", "gH", "gPace"]
for i in 1:8
    plot!(tt_long ./ 1000, res_rigorous[13+i, :], label=main_names[i])
end

plot!(tt_long ./ 1000, res_rigorous[30, :], label="gLNS")
plot!(tt_long ./ 1000, res_rigorous[31, :], label="gLCa")

plot!(tt_long ./ 1000, res_rigorous[34, :], label="gNtype", linestyle=:dash, linewidth=2)
plot!(tt_long ./ 1000, res_rigorous[35, :], label="gTtype", linestyle=:dash, linewidth=2)

ylabel!("g (mS/cm²)")
xlabel!("Time (s)")

# --- AFFICHAGE FINAL ---
l = @layout [a{0.25h}; b{0.25h}; c{0.5h}]
pt = plot(p_v_rig, p_ca_rig, p_gs_rig, layout=l, size=(1000, 1000), margin=10Plots.px)
display(pt)

# **Augmented Model - L-type Calcium Channel Blockade (Cav1.3) for Yu Model**

In [ ]:
#Définition des paramètres du controleur homeostatique
Ca_tgt = 0.00015    
tau_g  = 0.01 
t_Na  = 0.06 

gNa = 6.0
gCaL =0.139
gKd = 1.117
gKA = 1.68
gKERG = 0.13
gKSK = 0.07
gH = 0.078
gLNS = 0.28
gLCa = 0.00245
gNtype = 0.2
gTtype = 0.025

tCaL = t_Na * gNa / gCaL
tKd = t_Na * gNa/gKd 
tKA =  t_Na * gNa/ gKA
tKERG =  t_Na * gNa/ gKERG
tKSK =  t_Na * gNa/gKSK
tH =  t_Na * gNa/gH
tLNS =  t_Na * gNa/gLNS
tLCa =  t_Na * gNa/gLCa
tNCa = t_Na* gNa/ gNtype
tTCA = t_Na* gNa/ gTtype

Iapp_func(t) = 0.0  

p_homeo = (Iapp_func, Ca_tgt, tau_g,  t_Na, tCaL, tKd, tKA, tKERG, tKSK, tH, tLNS, tLCa, tNCa, tTCA)

V0 = 0.0
Ca0 = 1e-4 

u_biophys = [
    V0, m_inf(V0), h_inf(V0), hs_inf(V0), l_inf(V0), n_inf(V0), 
    p_inf(V0), q1_inf(V0), q2_inf(V0), 0.0, 0.0, mH_inf(V0), Ca0
]

g_init = [gNa, gCaL, gKd, gKA, gKERG, gKSK, gH, gLNS, gLCa]

m_init = copy(g_init)

g_cal = [gNtype, gTtype]

m_cal = copy(g_cal)

u_phys = [mN_inf(V0), hN_inf(V0), mT_inf(V0), hT_inf(V0)]

u0_homeo = vcat(u_biophys, g_init, m_init, g_cal, m_cal, u_phys)

T_long = 150000.0  
tspan_long = (0.0, T_long)

prob_2014 = ODEProblem(DA_homeo_2014_ODE_Ntype_Ttype_linfok, u0_homeo, tspan_long, p_homeo)

sol_2014 = solve(prob_2014, 
    RadauIIA5(),             
    reltol=1e-5,            
    abstol=1e-10,           
    maxiters=1e8, 
    isoutofdomain = (u,p,t) -> any(x -> x < 0, u[13:35]), 
    dtmax=1.0)           

if sol_2014.t[end] < T_long
    @warn "Crash persistant à t = $(sol_2014.t[end])."
end

# --- VISUALISATION TRACE ELECTRIQUE ---
tt_long = 0.0 : 10.0 : sol_2014.t[end]
res_2014 = sol_2014(tt_long)

t_zoom1 = 5000 : 0.1 : 7000
res_zoom1 = sol_2014(t_zoom1)
p_v1 = plot(t_zoom1 ./ 1e3, res_zoom1[1, :], linewidth=1.5, color=myDarkBlue, 
    legend=false, ylims=(-90, 30), xlims=(5, 7), 
    title="Electrical Trace 5-7 sec ",
    xticks=([5, 6, 7], ["5", "6", "7"]))
ylabel!("V (mV)")

t_zoom = 141000 : 0.1 : 143000 
res_zoom = sol_2014(t_zoom) 

p_v_2014 = plot(t_zoom ./ 1e3, res_zoom[1, :], linewidth=1.5, color=myDarkBlue, 
    legend=false, ylims=(-90, 30), xlims=(141, 143), 
    title="Electrical Trace 141-143 sec",
    xticks=([141, 142, 143], ["141", "142", "143"]))
ylabel!("V (mV)")


# --- VISUALISATION DU CALCIUM ---
ca_trace = res_2014[13, :]
ca_avg = moving_average(ca_trace, 300, 1) 
t_avg = range(0, 150, length=length(ca_avg))

p_ca = plot(t_avg, ca_avg, linewidth=2, color=:black, label="[Ca] moyen", title="Calcium Convergence")
hline!([Ca_tgt], color=:red, linestyle=:dash, label="Cible (Ca_tgt)")
ylabel!("[Ca]")

# --- VISUALISATION DES CONDUCTANCES ---
p_gs = plot(legend=:outerright, title="Conductance Evolution", yaxis=:log)
gCaL_eff = [ (t < 30.0) ? res_2014[15, i] : 1e-18 for (i, t) in enumerate(tt_long ./ 1000) ]
names_2014 = ["gNa", "gCaL", "gKd", "gKA", "gKERG", "gKSK", "gH", "gLNS", "gLCa"]

for i in [1, 3, 4, 5, 6, 7, 8, 9] # On saute i=2 (gCaL)
    plot!(tt_long ./ 1000, res_2014[13+i, :], label=names_2014[i])
end

plot!(tt_long ./ 1000, gCaL_eff, label="gCaL ", color=:red, linewidth=2)

plot!(tt_long ./ 1000, res_2014[32, :], label="gNtype", linewidth=2, linestyle=:dash)
plot!(tt_long ./ 1000, res_2014[33, :], label="gTtype", linewidth=2, linestyle=:dash)
ylims!(1e-6, 1e4)
ylabel!("g (mS/cm²)")
xlabel!("Time (s)")

# --- AFFICHAGE FINAL ---
l = @layout [a{0.25h}; b{0.25h}; c{0.25h}; d{0.25h}]
plot(p_v1, p_v_2014, p_ca, p_gs, layout=l, size=(1000, 1000), margin=10Plots.px) 

# **Augemented Model - L-type Calcium Channel Blockade (Cav1.3) for Fyon Model**

In [ ]:
#Définition des paramètres du controleur homeostatique
Ca_tgt = 7.84e-5    
tau_g  = 0.01     
t_Na  = 0.06    

gNa = 25
gCaL = 1.0
gKd = 10.0
gKA = 1.68
gKERG = 0.13
gKSK = 0.3
gH = 0.078
gPace = 5
gLNS = 0.01   
gLCa = 0.00245 
gNtype = 0.2
gTtype = 0.025

tCaL = t_Na * gNa/ gCaL
tKd = t_Na * gNa/gKd 
tKA =  t_Na * gNa/ gKA
tKERG =  t_Na * gNa/ gKERG
tKSK =  t_Na * gNa/gKSK
tH =  t_Na * gNa/gH
tLNS =  t_Na * gNa/gLNS
tLCa =  t_Na * gNa/gLCa
tPace = t_Na * gNa/gPace
tNCa = t_Na * gNa/gNtype
tTCA = t_Na * gNa/gTtype

Iapp_func(t) = 0.0  
     
p_homeo = (Iapp_func, Ca_tgt, tau_g, t_Na, tCaL, tKd, tKA, tKERG, tKSK, tH, tPace, tLNS, tLCa, tNCa, tTCA)

V0 = -50
Ca0 = 1e-4

u_biophys = [
    V0, m_inf_true(V0), h_inf(V0), hs_inf(V0), l_inf_true(V0), n_inf(V0), 
    p_inf(V0), q1_inf(V0), q2_inf(V0), 0.0, 0.0, mH_inf(V0), Ca0
]

g_main_init = [gNa, gCaL, gKd, gKA, gKERG, gKSK, gH,gPace]

m_main_init = copy(g_main_init)

g_leak_init = [gLNS,  gLCa]

m_leak_init = copy(g_leak_init)

g_calcique = [gNtype, gTtype]

m_calcique = copy(g_calcique)

uphys = [mN_inf(V0), hN_inf(V0), mT_inf(V0), hT_inf(V0)]

u0_homeo = vcat(u_biophys, g_main_init, m_main_init, g_leak_init, m_leak_init, g_calcique, m_calcique, uphys)

T_long = 300000.0 
tspan_long = (0.0, T_long)

prob_rigorous = ODEProblem(DA_homeo_ODE_Ntype_Ttype_gpace_Ltype, u0_homeo, tspan_long, p_homeo)

sol_rigorous = solve(prob_rigorous, 
    KenCarp47(),             
    reltol=1e-5,            
    abstol=1e-8,            
    maxiters=1e9, 
    isoutofdomain = (u,p,t) -> any(x -> x < 0, u[13:37]), s
    dtmax=1.0)            
if sol_rigorous.t[end] < T_long
    @warn "Crash persistant à t = $(sol_rigorous.t[end])"
end

# --- VISUALISATION TRACE ELECTRIQUE ---
tt_long = 0.0 : 10.0 : sol_rigorous.t[end]
res_rigorous = sol_rigorous(tt_long)

t_zoom1 = 5000 : 0.1 : 7000
res_zoom1 = sol_rigorous(t_zoom1)
p_v1 = plot(t_zoom1 ./ 1e3, res_zoom1[1, :], linewidth=1.5, color=myDarkBlue, 
    legend=false, ylims=(-90, 30), xlims=(5, 7), 
    title="Electrical Trace 5-7 sec ",
    xticks=([5, 6, 7], ["5", "6", "7"]))
ylabel!("V (mV)")

t_zoom = 291000 : 0.1 : 293000 
res_zoom = sol_rigorous(t_zoom) 

p_v_rig = plot(t_zoom ./ 1e3, res_zoom[1, :], linewidth=1.5, color=myDarkBlue, 
    legend=false, ylims=(-90, 30), xlims=(291, 293), 
    title="Electrical Trace 291-293 sec",
    xticks=([291, 292, 293], ["291", "292", "293"]))
ylabel!("V (mV)")

# --- VISUALISATION DU CALCIUM ---
ca_trace_rig = res_rigorous[13, :]
ca_avg_rig = moving_average(ca_trace_rig, 300, 1)
t_avg_rig = range(0, 300, length=length(ca_avg_rig))

p_ca_rig = plot(t_avg_rig, ca_avg_rig, linewidth=2, color=:black, label="[Ca] moyen", title="Calcium Convergence")
hline!([Ca_tgt], color=:red, linestyle=:dash, label="Ca_tgt")
ylabel!("[Ca]")

# --- VISUALISATION DES CONDUCTANCES ---
p_gs_rig = plot(legend=:outerright, title="Conductance Evolution", yaxis=:log)
ts_sec = tt_long ./ 1000
main_names = ["gNa", "gCaL", "gKd", "gKA", "gKERG", "gKSK", "gH", "gPace"]

for i in [1, 3, 4, 5, 6, 7, 8]
    plot!(ts_sec, res_rigorous[13+i, :], label=main_names[i])
end

gCaL_eff = [ (t < 30.0) ? res_rigorous[15, i] : 1e-18 for (i, t) in enumerate(ts_sec) ]
plot!(ts_sec, gCaL_eff, label="gCaL", 
      color=:red, linewidth=2.5)

plot!(ts_sec, res_rigorous[30, :], label="gLNS", alpha=0.7)
plot!(ts_sec, res_rigorous[31, :], label="gLCa", alpha=0.7)

plot!(ts_sec, res_rigorous[34, :], label="gNtype", linestyle=:dash, linewidth=2.5, color=:black)
plot!(ts_sec, res_rigorous[35, :], label="gTtype", linestyle=:dash, linewidth=2.5, color=:purple)
ylims!(1e-6, 1e4)
ylabel!("g (mS/cm²)")
xlabel!("Time (s)")

# --- AFFICHAGE FINAL ---
l = @layout [a{0.25h}; b{0.25h}; c{0.25h}; d{0.25h}]
plot(p_v1, p_v_rig, p_ca_rig, p_gs_rig, layout=l, size=(1000, 1000), margin=10Plots.px)

# **Augmented Model -- gPace Conductance Blockade for Fyon Model**

In [ ]:
#Définition des paramètres du controleur homeostatique
Ca_tgt = 7.84e-5    
tau_g  = 0.01    
t_Na  = 0.06    

gNa = 25
gCaL = 1.0
gKd = 10.0
gKA = 1.68
gKERG = 0.13
gKSK = 0.3
gH = 0.078
gPace = 5
gLNS = 0.01   
gLCa = 0.00245 
gNtype = 0.2
gTtype = 0.025

tCaL = t_Na * gNa/ gCaL
tKd = t_Na * gNa/gKd 
tKA =  t_Na * gNa/ gKA
tKERG =  t_Na * gNa/ gKERG
tKSK =  t_Na * gNa/gKSK
tH =  t_Na * gNa/gH
tLNS =  t_Na * gNa/gLNS
tLCa =  t_Na * gNa/gLCa
tPace = t_Na * gNa/gPace
tNCa = t_Na * gNa/gNtype
tTCA = t_Na * gNa/gTtype

Iapp_func(t) = 0.0  
     
p_homeo = (Iapp_func, Ca_tgt, tau_g, t_Na, tCaL, tKd, tKA, tKERG, tKSK, tH, tPace, tLNS, tLCa, tNCa, tTCA)

V0 = -50
Ca0 = 7.81e-5

u_biophys = [
    V0, m_inf_true(V0), h_inf(V0), hs_inf(V0), l_inf_true(V0), n_inf(V0), 
    p_inf(V0), q1_inf(V0), q2_inf(V0), 0.0, 0.0, mH_inf(V0), Ca0
]

g_main_init = [gNa, gCaL, gKd, gKA, gKERG, gKSK, gH,gPace]

m_main_init = copy(g_main_init)

g_leak_init = [gLNS,  gLCa]

m_leak_init = copy(g_leak_init)

g_calcique = [gNtype, gTtype]

m_calcique = copy(g_calcique)

uphys = [mN_inf(V0), hN_inf(V0), mT_inf(V0), hT_inf(V0)]

u0_homeo = vcat(u_biophys, g_main_init, m_main_init, g_leak_init, m_leak_init, g_calcique, m_calcique, uphys)

T_long = 300000.0 
tspan_long = (0.0, T_long)

prob_rigorous = ODEProblem(test_gpace, u0_homeo, tspan_long, p_homeo)

sol_rigorous = solve(prob_rigorous, 
    KenCarp47(),             
    reltol=1e-5,            
    abstol=1e-8,            
    maxiters=1e9, 
    isoutofdomain = (u,p,t) -> any(x -> x < 0, u[13:37]), 
    dtmax=1.0)               
if sol_rigorous.t[end] < T_long
    @warn "Crash persistant à t = $(sol_rigorous.t[end])"
end

t_end = sol_rigorous.t[end]

if t_end < T_long
    @warn "Simulation interrompue prématurément à t = $(round(t_end/1000, digits=2)) s."
end


# --- VISUALISATION TRACE ELECTRIQUE ---
tt_long = 0.0 : 10.0 : t_end
res_rigorous = sol_rigorous(tt_long)

t_zoom1 = 5000 : 0.1 : 7000
res_zoom1 = sol_rigorous(t_zoom1)
p_v1 = plot(t_zoom1 ./ 1e3, res_zoom1[1, :], linewidth=1.5, color=myDarkBlue, 
    legend=false, ylims=(-90, 30), xlims=(5, 7), 
    title="Electrical Trace 5-7 sec ",
    xticks=([5, 6, 7], ["5", "6", "7"]))
ylabel!("V (mV)")

t_zoom = 291000 : 0.1 : 293000 
res_zoom = sol_rigorous(t_zoom) 

p_v = plot(t_zoom ./ 1e3, res_zoom[1, :], linewidth=1.5, color=myDarkBlue, 
    legend=false, ylims=(-90, 30), xlims=(291, 293), 
    title="Electrical Trace 291-293 sec",
    xticks=([291,292,293], ["291", "292", "293"]))
ylabel!("V (mV)")


# --- VISUALISATION DU CALCIUM ---
ca_trace_rig = res_rigorous[13, :]
ca_avg_rig = moving_average(ca_trace_rig, 300, 1)

t_avg_rig = range(0, t_end/1000, length=length(ca_avg_rig))

p_ca_rig = plot(t_avg_rig, ca_avg_rig, linewidth=2, color=:black, 
    label="[Ca] moyen", title="Calcium Convergence")
hline!([Ca_tgt], color=:red, linestyle=:dash, label="Cible")
ylabel!("[Ca]")

# --- VISUALISATION DES CONDUCTANCES ---
p_gs_rig = plot(legend=:outerright, title="Conductances Evolution", yaxis=:log)
ts_sec = tt_long ./ 1000

main_names = ["gNa", "gCaL", "gKd", "gKA", "gKERG", "gKSK", "gH", "gPace"]

for i in 1:7 
    plot!(ts_sec, res_rigorous[13+i, :], label=main_names[i])
end

gPace_eff = [ (t < 10.0) ? res_rigorous[21, i] : 1e-18 for (i, t) in enumerate(ts_sec) ]
plot!(ts_sec, gPace_eff, label="gPace", 
      color=:red, linewidth=2.5)

plot!(ts_sec, res_rigorous[30, :], label="gLNS", alpha=0.6)
plot!(ts_sec, res_rigorous[31, :], label="gLCa", alpha=0.6)

plot!(ts_sec, res_rigorous[34, :], label="gNtype", linestyle=:dash, linewidth=2.5, color=:black)
plot!(ts_sec, res_rigorous[35, :], label="gTtype", linestyle=:dash, linewidth=2.5, color=:purple)

ylabel!(" g mS/cm²")
xlabel!("Time (s)")
ylims!(1e-6, 1e4) 

# --- 6. AFFICHAGE FINAL ---
l = @layout [a{0.25h}; b{0.25h}; c{0.25h}; d{0.25h}]
final_plot = plot(p_v1, p_v, p_ca_rig, p_gs_rig, layout=l, size=(1000, 1000), margin=15Plots.px)

display(final_plot)

# **Fixed Calcium Leak + Augmented Models + L-type Calcium Channel Blockade (Cav1.3) for Yu Model**
**Objective:** gLCa decoupled from homeostatic loop (fixed at baseline). 

In [ ]:
#Définition des paramètres du controleur homeostatique
Ca_tgt = 0.00015    
tau_g  = 0.01 
t_Na  = 0.06 

gNa = 6.0
gCaL =0.139
gKd = 1.117
gKA = 1.68
gKERG = 0.13
gKSK = 0.07
gH = 0.078
gLNS = 0.28
gLCa = 0.00245
gNtype = 0.2
gTtype = 0.025

tCaL = t_Na * gNa / gCaL
tKd = t_Na * gNa/gKd 
tKA =  t_Na * gNa/ gKA
tKERG =  t_Na * gNa/ gKERG
tKSK =  t_Na * gNa/gKSK
tH =  t_Na * gNa/gH
tLNS =  t_Na * gNa/gLNS
tLCa =  t_Na * gNa/gLCa
tNCa = t_Na* gNa/ gNtype
tTCA = t_Na* gNa/ gTtype

Iapp_func(t) = 0.0  # pA


p_homeo = (Iapp_func, Ca_tgt, tau_g,  t_Na, tCaL, tKd, tKA, tKERG, tKSK, tH, tLNS, tLCa, tNCa, tTCA)

V0 = 0.0
Ca0 = 1e-4 

u_biophys = [
    V0, m_inf(V0), h_inf(V0), hs_inf(V0), l_inf(V0), n_inf(V0), 
    p_inf(V0), q1_inf(V0), q2_inf(V0), 0.0, 0.0, mH_inf(V0), Ca0
]

g_init = [gNa, gCaL, gKd, gKA, gKERG, gKSK, gH, gLNS, gLCa]

m_init = copy(g_init)

g_cal = [gNtype, gTtype]

m_cal = copy(g_cal)

u_phys = [mN_inf(V0), hN_inf(V0), mT_inf(V0), hT_inf(V0)]

u0_homeo = vcat(u_biophys, g_init, m_init, g_cal, m_cal, u_phys)

T_long = 800000.0  
tspan_long = (0.0, T_long)

prob_2014 = ODEProblem(DA_homeo_2014_ODE_Ntype_Ttype_Ltypeblock_LeakC, u0_homeo, tspan_long, p_homeo)

sol_2014 = solve(prob_2014, 
    RadauIIA5(),            
    reltol=1e-5,             
    abstol=1e-10,            
    maxiters=1e8, 
    isoutofdomain = (u,p,t) -> any(x -> x < 0, u[13:35]), 
    dtmax=1.0)               

if sol_2014.t[end] < T_long
    @warn "Crash persistant à t = $(sol_2014.t[end])."
end



# --- VISUALISATION TRACE ELECTRIQUE ---
tt_long = 0.0 : 10.0 : sol_2014.t[end]
res_2014 = sol_2014(tt_long)

t_zoom1 = 5000 : 0.1 : 7000
res_zoom1 = sol_2014(t_zoom1)
p_v1 = plot(t_zoom1 ./ 1e3, res_zoom1[1, :], linewidth=1.5, color=myDarkBlue, 
    legend=false, ylims=(-90, 30), xlims=(5, 7), 
    title="Electrical Trace 5-7 sec ",
    xticks=([5, 6, 7], ["5", "6", "7"]))
ylabel!("V (mV)")

t_zoom = 791000 : 0.1 : 793000 
res_zoom = sol_2014(t_zoom) 

p_v_2014 = plot(t_zoom ./ 1e3, res_zoom[1, :], linewidth=1.5, color=myDarkBlue, 
    legend=false, ylims=(-90, 30), xlims=(791, 793), 
    title="Electrical Trace 791-792 sec",
    xticks=([791, 792, 793], ["791", "792", "793"]))
ylabel!("V (mV)")


# --- VISUALISATION DU CALCIUM ---
ca_trace = res_2014[13, :]
ca_avg = moving_average(ca_trace, 300, 1) 
t_avg = range(0, 700, length=length(ca_avg))

p_ca = plot(t_avg, ca_avg, linewidth=2, color=:black, label="[Ca] moyen", title="Calcium Convergence",
legend = :topright)
hline!([Ca_tgt], color=:red, linestyle=:dash, label="Cible (Ca_tgt)")
ylabel!("[Ca]")
xlabel!("Temps (s)")

# --- VISUALISATION DES CONDUCTANCES ---
p_gs = plot(legend=:outerright, title="Conductance Evolution", yaxis=:log)
gCaL_eff = [ (t < 30.0) ? res_2014[15, i] : 1e-18 for (i, t) in enumerate(tt_long ./ 1000) ]
names_2014 = ["gNa", "gCaL", "gKd", "gKA", "gKERG", "gKSK", "gH", "gLNS", "gLCa"]

for i in [1, 3, 4, 5, 6, 7, 8, 9] 
    plot!(tt_long ./ 1000, res_2014[13+i, :], label=names_2014[i])
end

plot!(tt_long ./ 1000, gCaL_eff, label="gCaL ", color=:red, linewidth=2)
plot!(tt_long ./ 1000, res_2014[32, :], label="gNtype", linewidth=2, linestyle=:dash)
plot!(tt_long ./ 1000, res_2014[33, :], label="gTtype", linewidth=2, linestyle=:dash)

ylabel!("g (mS/cm²)")
xlabel!("Time (s)")
ylims!(1e-6, 1e4)

# --- AFFICHAGE FINAL ---
l = @layout [a{0.25h}; b{0.25h}; c{0.25h}; d{0.25h}]
plot(p_v1, p_v_2014, p_ca, p_gs, layout=l, size=(1000, 1000), margin=10Plots.px) 

# **Fixed Calcium Leak + Augmented Models + L-type Calcium Channel Blockade (Cav1.3) for Fyon Model**
**Objective:** gLCa decoupled from homeostatic loop (fixed at baseline). 

In [ ]:
#Définition des paramètres du controleur homeostatique
Ca_tgt = 7.81e-5    
tau_g  = 0.01     
t_Na  = 0.06    


gNa = 25
gCaL = 1.0
gKd = 10.0
gKA = 1.68
gKERG = 0.13
gKSK = 0.3
gH = 0.078
gPace = 5
gLNS = 0.01   
gLCa = 0.00245 
gNtype = 0.2
gTtype = 0.025

tCaL = t_Na * gNa/ gCaL
tKd = t_Na * gNa/gKd 
tKA =  t_Na * gNa/ gKA
tKERG =  t_Na * gNa/ gKERG
tKSK =  t_Na * gNa/gKSK
tH =  t_Na * gNa/gH
tLNS =  t_Na * gNa/gLNS
tLCa =  t_Na * gNa/gLCa
tPace = t_Na * gNa/gPace
tNCa = t_Na * gNa/gNtype
tTCA = t_Na * gNa/gTtype

Iapp_func(t) = 0.0  
     
p_homeo = (Iapp_func, Ca_tgt, tau_g, t_Na, tCaL, tKd, tKA, tKERG, tKSK, tH, tPace, tLNS, tLCa, tNCa, tTCA)

V0 = -50
Ca0 = 1e-4

u_biophys = [
    V0, m_inf_true(V0), h_inf(V0), hs_inf(V0), l_inf_true(V0), n_inf(V0), 
    p_inf(V0), q1_inf(V0), q2_inf(V0), 0.0, 0.0, mH_inf(V0), Ca0
]

g_main_init = [gNa, gCaL, gKd, gKA, gKERG, gKSK, gH,gPace]

m_main_init = copy(g_main_init)

g_leak_init = [gLNS,  gLCa]

m_leak_init = copy(g_leak_init)

g_calcique = [gNtype, gTtype]

m_calcique = copy(g_calcique)

uphys = [mN_inf(V0), hN_inf(V0), mT_inf(V0), hT_inf(V0)]

u0_homeo = vcat(u_biophys, g_main_init, m_main_init, g_leak_init, m_leak_init, g_calcique, m_calcique, uphys)

T_long = 500000.0 
tspan_long = (0.0, T_long)

prob_rigorous = ODEProblem(DA_homeo_ODE_Ntype_Ttype_FixedLeak_Ltype, u0_homeo, tspan_long, p_homeo)

sol_rigorous = solve(prob_rigorous, 
    KenCarp47(),             
    reltol=1e-5,            
    abstol=1e-8,           
    maxiters=1e9, 
    isoutofdomain = (u,p,t) -> any(x -> x < 0, u[13:37]), 
    dtmax=1.0)              
if sol_rigorous.t[end] < T_long
    @warn "Crash persistant à t = $(sol_rigorous.t[end])"
end


tt_long = 0.0 : 10.0 : sol_rigorous.t[end]
res_rigorous = sol_rigorous(tt_long)

t_zoom1 = 5000 : 0.1 : 7000
res_zoom1 = sol_rigorous(t_zoom1)
p_v1 = plot(t_zoom1 ./ 1e3, res_zoom1[1, :], linewidth=1.5, color=myDarkBlue, 
    legend=false, ylims=(-90, 30), xlims=(5, 7), 
    title="Electrical Trace 5-7 sec",
    xticks=([5, 6, 7], ["5", "6", "7"]))
ylabel!("V (mV)")

# --- VISUALISATION TRACE ELECTRIQUE ---
t_zoom = 491000 : 0.1 : 493000 
res_zoom = sol_rigorous(t_zoom) 
p_v_rig = plot(t_zoom ./ 1e3, res_zoom[1, :], linewidth=1.5, color=myDarkBlue, 
    legend=false, ylims=(-90, 30), xlims=(491, 493), 
    title="Electrical Trace 491-492 sec",
    xticks=([491, 492, 493], ["491", "492", "493"]))
ylabel!("V (mV)")

# --- VISUALISATION DU CALCIUM ---
ca_trace_rig = res_rigorous[13, :]
ca_avg_rig = moving_average(ca_trace_rig, 300, 1)
t_avg_rig = range(0, 500, length=length(ca_avg_rig))

p_ca_rig = plot(t_avg_rig, ca_avg_rig, linewidth=2, color=:black, label="[Ca] moyen", title="Calcium Convergence",
    legend = :topright)
hline!([Ca_tgt], color=:red, linestyle=:dash, label="Ca_tgt")
ylabel!("[Ca]")

# --- VISUALISATION DES CONDUCTANCES ---
p_gs_rig = plot(legend=:outerright, title="Conductance Evolution", yaxis=:log)
ts_sec = tt_long ./ 1000
main_names = ["gNa", "gCaL", "gKd", "gKA", "gKERG", "gKSK", "gH", "gPace"]

for i in [1, 3, 4, 5, 6, 7, 8] 
    plot!(ts_sec, res_rigorous[13+i, :], label=main_names[i])
end

gCaL_eff = [ (t < 30.0) ? res_rigorous[15, i] : 1e-18 for (i, t) in enumerate(ts_sec) ]
plot!(ts_sec, gCaL_eff, label="gCaL", 
      color=:red, linewidth=2.5)

plot!(ts_sec, res_rigorous[30, :], label="gLNS", alpha=0.7)
plot!(ts_sec, res_rigorous[31, :], label="gLCa", alpha=0.7)
plot!(ts_sec, res_rigorous[34, :], label="gNtype", linestyle=:dash, linewidth=2.5, color=:black)
plot!(ts_sec, res_rigorous[35, :], label="gTtype", linestyle=:dash, linewidth=2.5, color=:purple)

ylabel!("g (mS/cm²)")
xlabel!("Time (s)")
ylims!(1e-6, 1e4) 

# --- AFFICHAGE FINAL ---
l = @layout [a{0.25h}; b{0.25h}; c{0.25h}; d{0.25h}]
plot(p_v1, p_v_rig, p_ca_rig, p_gs_rig, layout=l, size=(1000, 1000), margin=10Plots.px)

# **Fixed Calcium Leak + Augmented Models + gPace Blockade for Fyon Model**

In [ ]:
#Définition des paramètres du controleur homeostatique
Ca_tgt = 7.81e-5    
tau_g  = 0.01    
t_Na  = 0.06    

gNa = 25
gCaL = 1.0
gKd = 10.0
gKA = 1.68
gKERG = 0.13
gKSK = 0.3
gH = 0.078
gPace = 5
gLNS = 0.01   
gLCa = 0.00245 
gNtype = 0.2
gTtype = 0.025

tCaL = t_Na * gNa/ gCaL
tKd = t_Na * gNa/gKd 
tKA =  t_Na * gNa/ gKA
tKERG =  t_Na * gNa/ gKERG
tKSK =  t_Na * gNa/gKSK
tH =  t_Na * gNa/gH
tLNS =  t_Na * gNa/gLNS
tLCa =  t_Na * gNa/gLCa
tPace = t_Na * gNa/gPace
tNCa = t_Na * gNa/gNtype
tTCA = t_Na * gNa/gTtype

Iapp_func(t) = 0.0  

p_homeo = (Iapp_func, Ca_tgt, tau_g, t_Na, tCaL, tKd, tKA, tKERG, tKSK, tH, tPace, tLNS, tLCa, tNCa, tTCA)

V0 = -50
Ca0 = 7.81e-5

u_biophys = [
    V0, m_inf_true(V0), h_inf(V0), hs_inf(V0), l_inf_true(V0), n_inf(V0), 
    p_inf(V0), q1_inf(V0), q2_inf(V0), 0.0, 0.0, mH_inf(V0), Ca0
]

g_main_init = [gNa, gCaL, gKd, gKA, gKERG, gKSK, gH,gPace]

m_main_init = copy(g_main_init)

g_leak_init = [gLNS,  gLCa]

m_leak_init = copy(g_leak_init)

g_calcique = [gNtype, gTtype]

m_calcique = copy(g_calcique)

uphys = [mN_inf(V0), hN_inf(V0), mT_inf(V0), hT_inf(V0)]

u0_homeo = vcat(u_biophys, g_main_init, m_main_init, g_leak_init, m_leak_init, g_calcique, m_calcique, uphys)

T_long = 200000.0 
tspan_long = (0.0, T_long)

prob_rigorous = ODEProblem(DA_homeo_ODE_Ntype_Ttype_Ltype_leakC, u0_homeo, tspan_long, p_homeo)

sol_rigorous = solve(prob_rigorous, 
    KenCarp47(),             
    reltol=1e-5,             
    abstol=1e-8,      
    maxiters=1e9, 
    isoutofdomain = (u,p,t) -> any(x -> x < 0, u[13:37]), 
    dtmax=1.0)               
if sol_rigorous.t[end] < T_long
    @warn "Crash persistant à t = $(sol_rigorous.t[end])"
end

# On récupère la fin RÉELLE de la simulation
t_end = sol_rigorous.t[end]

if t_end < T_long
    @warn "Simulation interrompue prématurément à t = $(round(t_end/1000, digits=2)) s."
end

# --- VISUALISATION TRACE ELECTRIQUE ---
tt_long = 0.0 : 10.0 : t_end
res_rigorous = sol_rigorous(tt_long)

t_zoom1 = 5000 : 0.1 : 7000
res_zoom1 = sol_rigorous(t_zoom1)
p_v1 = plot(t_zoom1 ./ 1e3, res_zoom1[1, :], linewidth=1.5, color=myDarkBlue, 
    legend=false, ylims=(-90, 30), xlims=(5, 7), 
    title="Electrical Trace 5-7 sec ",
    xticks=([5, 6, 7], ["5", "6", "7"]))
ylabel!("V (mV)")

t_zoom = 41000 : 0.1 : 43000 
res_zoom = sol_rigorous(t_zoom) 

p_v = plot(t_zoom ./ 1e3, res_zoom[1, :], linewidth=1.5, color=myDarkBlue, 
    legend=false, ylims=(-90, 30), xlims=(41, 43), 
    title="Electrical Trace 41-43 sec",
    xticks=([41, 42, 43], ["41", "42", "43"]))
ylabel!("V (mV)")

t_zoom3 = 81000 : 0.1 : 83000 
res_zoom3 = sol_rigorous(t_zoom3) 

p_v3 = plot(t_zoom3 ./ 1e3, res_zoom3[1, :], linewidth=1.5, color=myDarkBlue, 
    legend=false, ylims=(-90, 30), xlims=(81, 83), 
    title="Electrical Trace 81-83 sec",
    xticks=([81, 82, 83], ["81", "82", "83"]))
ylabel!("V (mV)")

ca_trace_rig = res_rigorous[13, :]
ca_avg_rig = moving_average(ca_trace_rig, 300, 1)

t_avg_rig = range(0, t_end/1000, length=length(ca_avg_rig))

p_ca_rig = plot(t_avg_rig, ca_avg_rig, linewidth=2, color=:black, 
    label="[Ca] moyen", title="Calcium Convergence")
hline!([Ca_tgt], color=:red, linestyle=:dash, label="Cible")
ylabel!("[Ca]")

# --- VISUALISATION DES CONDUCTANCES  ---
p_gs_rig = plot(legend=:outerright, title="Conductance Evolution", yaxis=:log)

ts_sec = tt_long ./ 1000

main_names = ["gNa", "gCaL", "gKd", "gKA", "gKERG", "gKSK", "gH"]

for i in [1,2, 3, 4, 5, 6, 7] 
    plot!(ts_sec, res_rigorous[13+i, :], label=main_names[i])
end

gPace_eff = [ (t < 10.0) ? res_rigorous[21, i] : 1e-18 for (i, t) in enumerate(ts_sec) ]
plot!(ts_sec, gPace_eff, label="gPace", 
      color=:red, linewidth=2.5)

plot!(ts_sec, res_rigorous[30, :], label="gLNS", alpha=0.6)
plot!(ts_sec, res_rigorous[31, :], label="gLCa", alpha=0.6)

plot!(ts_sec, res_rigorous[34, :], label="gNtype", linestyle=:dash, linewidth=2.5, color=:black)
plot!(ts_sec, res_rigorous[35, :], label="gTtype", linestyle=:dash, linewidth=2.5, color=:purple)

ylabel!("g(mS/cm²)")
xlabel!("Time (s)")
ylims!(1e-6, 1e4)

# --- 6. AFFICHAGE FINAL ---
l = @layout [a{0.33h}; b{0.33h}; c{0.34h}]
#final_plot = plot(p_v1, p_v, p_v3, p_ca_rig, p_gs_rig, layout=l, size=(1000, 1000), margin=15Plots.px)
final_plot = plot(p_v1, p_v, p_v3, layout=l, size=(1000, 1000), margin=15Plots.px)
display(final_plot)